# 07 - Python Toolbox Diagnostics

This notebook covers toolbox scripts, manual python-toolbox enable/disable controls, in-cluster diagnostics, memory usage, and GPU correlation.

Your current local profile keeps `pythonToolbox.enabled: true` by default so in-cluster diagnostics are available immediately. Use the manual toggle below only if you want to switch it off temporarily.


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    except Exception as exc:
        print(f"Package install failed: {exc}")
        print("Continue with notebook cells that do not depend on missing packages.")
else:
    print("All common packages are already installed.")


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path


PROJECT_ROOT = Path("/media/waqasm86/External1/Project-Nvidia-Office/Project-Llamatelemetry/langchain-kubernetes-jupyterlab/llm-observability-stack").resolve()
NAMESPACE = "llm-observability"
CWD = Path.cwd().resolve()


def is_within_project_root(path: Path, root: Path) -> bool:
    try:
        path = path.resolve()
        root = root.resolve()
        return path == root or root in path.parents
    except Exception:
        return False


PROJECT_SCOPE_OK = is_within_project_root(CWD, PROJECT_ROOT)


def run(cmd: str, check: bool = True, quiet: bool = False, cwd: Path | None = None):
    display_cmd = cmd if cwd is None else f"(cd {shlex.quote(str(cwd))} && {cmd})"
    if not quiet:
        print(f"$ {display_cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
        cwd=str(cwd) if cwd is not None else None,
    )
    if not quiet and proc.stdout:
        print(proc.stdout.rstrip())
    if not quiet and proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        stderr = proc.stderr.strip()
        raise RuntimeError(f"Command failed ({proc.returncode}): {display_cmd}\n{stderr}")
    return proc


def run_json(cmd: str, check: bool = True):
    proc = run(cmd, check=check, quiet=True)
    if proc.returncode != 0 or not proc.stdout.strip():
        return {}
    try:
        return json.loads(proc.stdout)
    except json.JSONDecodeError:
        return {}


def can_run_toolbox(action: str = "python-toolbox diagnostics") -> bool:
    if not PROJECT_SCOPE_OK:
        print(f"Skipping {action}: current working directory is outside project root.")
        print(f"  cwd         : {CWD}")
        print(f"  project root: {PROJECT_ROOT}")
        return False
    if shutil.which("kubectl") is None:
        print(f"Skipping {action}: kubectl is not available in PATH.")
        return False
    return True


def can_run_helm(action: str = "helm command") -> bool:
    if not PROJECT_SCOPE_OK:
        print(f"Skipping {action}: current working directory is outside project root.")
        print(f"  cwd         : {CWD}")
        print(f"  project root: {PROJECT_ROOT}")
        return False
    if shutil.which("helm") is None:
        print(f"Skipping {action}: helm is not available in PATH.")
        return False
    return True


def run_kubectl(args: str, check: bool = True, quiet: bool = False):
    if not can_run_toolbox("kubectl command"):
        return subprocess.CompletedProcess(
            args=f"kubectl {args}",
            returncode=126,
            stdout="",
            stderr="project scope or kubectl precheck failed",
        )
    return run(f"kubectl {args}", check=check, quiet=quiet)


def run_kubectl_json(args: str, check: bool = True):
    proc = run_kubectl(args, check=check, quiet=True)
    if proc.returncode != 0 or not proc.stdout.strip():
        return {}
    try:
        return json.loads(proc.stdout)
    except json.JSONDecodeError:
        return {}


def list_toolbox_pods() -> list[str]:
    pod_data = run_kubectl_json(
        f"get pods -n {NAMESPACE} -l app.kubernetes.io/name=python-toolbox -o json",
        check=False,
    )
    return [
        item.get("metadata", {}).get("name", "")
        for item in pod_data.get("items", [])
        if item.get("metadata", {}).get("name")
    ]


def list_running_toolbox_pods() -> list[str]:
    pod_data = run_kubectl_json(
        f"get pods -n {NAMESPACE} -l app.kubernetes.io/name=python-toolbox -o json",
        check=False,
    )
    return [
        item.get("metadata", {}).get("name", "")
        for item in pod_data.get("items", [])
        if item.get("metadata", {}).get("name") and item.get("status", {}).get("phase") == "Running"
    ]


def parse_manual_toggle(value: str) -> bool:
    normalized = str(value).strip().lower()
    truthy = {"on", "yes", "true", "1"}
    falsy = {"off", "no", "false", "0"}
    if normalized in truthy:
        return True
    if normalized in falsy:
        return False
    raise ValueError(
        f"Unsupported toggle value: {value!r}. Use on/off, yes/no, true/false, or 1/0."
    )


def build_python_toolbox_helm_command(enabled: bool, values_file: Path) -> str:
    enabled_literal = "true" if enabled else "false"
    return (
        f"helm upgrade --install llm-observability-stack . "
        f"-n {NAMESPACE} --create-namespace "
        f"-f {shlex.quote(str(values_file))} "
        f"--set pythonToolbox.enabled={enabled_literal}"
    )


print(f"Project root    : {PROJECT_ROOT}")
print(f"Current cwd     : {CWD}")
print(f"Project scope OK: {PROJECT_SCOPE_OK}")


## Check Current Toolbox Status


In [ ]:
import yaml

values_path = PROJECT_ROOT / "values.local-k3s.yaml"
if values_path.exists():
    values = yaml.safe_load(values_path.read_text()) or {}
else:
    values = {}
    print(f"values.local-k3s.yaml not found at: {values_path}")

file_enabled = values.get("pythonToolbox", {}).get("enabled")
print("values.local-k3s.yaml pythonToolbox.enabled:", file_enabled)

release_enabled = None
if can_run_helm("helm release value check"):
    release_proc = run(
        f"helm get values llm-observability-stack -n {NAMESPACE} -a",
        check=False,
        quiet=True,
        cwd=PROJECT_ROOT,
    )
    if release_proc.returncode == 0 and release_proc.stdout.strip():
        try:
            release_values = yaml.safe_load(release_proc.stdout) or {}
            release_enabled = release_values.get("pythonToolbox", {}).get("enabled")
        except Exception as exc:
            print(f"Unable to parse Helm release values: {exc}")
    else:
        print("Helm release values unavailable. Is llm-observability-stack installed?")

print("live Helm release pythonToolbox.enabled   :", release_enabled)

if can_run_toolbox("python-toolbox status checks"):
    deploy_obj = run_kubectl_json(f"get deploy -n {NAMESPACE} python-toolbox -o json", check=False)
    if deploy_obj:
        replicas = deploy_obj.get("spec", {}).get("replicas", 0)
        ready = deploy_obj.get("status", {}).get("readyReplicas", 0)
        print(f"python-toolbox deployment: present (spec.replicas={replicas}, readyReplicas={ready})")
    else:
        print("python-toolbox deployment: not found")

    pod_obj = run_kubectl_json(
        f"get pods -n {NAMESPACE} -l app.kubernetes.io/name=python-toolbox -o json",
        check=False,
    )
    pod_items = pod_obj.get("items", [])
    if not pod_items:
        print("python-toolbox pods: none")
    else:
        pod_names = [item.get("metadata", {}).get("name", "") for item in pod_items]
        print("python-toolbox pods:", ", ".join([name for name in pod_names if name]))


## Manually Enable or Disable Python Toolbox

Use `PYTHON_TOOLBOX_TOGGLE = "on"` or `"off"` below. The cell also accepts `yes`/`no`, `true`/`false`, and `1`/`0`.

Keep `APPLY_PYTHON_TOOLBOX_TOGGLE = False` for preview mode. Set it to `True` only when you want the notebook to run Helm for you.


In [ ]:
# Manual toggle: accepts on/off, yes/no, true/false, or 1/0.
import yaml

PYTHON_TOOLBOX_TOGGLE = "on"
APPLY_PYTHON_TOOLBOX_TOGGLE = False
WAIT_FOR_TOOLBOX_ROLLOUT = True
TOOLBOX_ROLLOUT_TIMEOUT_SECONDS = 180

values_file = PROJECT_ROOT / "values.local-k3s.yaml"
toggle_values = yaml.safe_load(values_file.read_text()) or {} if values_file.exists() else {}
desired_toolbox_enabled = parse_manual_toggle(PYTHON_TOOLBOX_TOGGLE)
current_values_enabled = toggle_values.get("pythonToolbox", {}).get("enabled")

print(f"Current values.local-k3s.yaml pythonToolbox.enabled: {current_values_enabled}")
print(f"Desired notebook toggle                    : {desired_toolbox_enabled}")

if not values_file.exists():
    print(f"values.local-k3s.yaml not found at: {values_file}")
elif can_run_helm("python-toolbox helm toggle"):
    helm_cmd = build_python_toolbox_helm_command(desired_toolbox_enabled, values_file)
    print("Helm command:")
    print(helm_cmd)

    if APPLY_PYTHON_TOOLBOX_TOGGLE:
        run(helm_cmd, cwd=PROJECT_ROOT)
        if WAIT_FOR_TOOLBOX_ROLLOUT and desired_toolbox_enabled:
            run_kubectl(
                f"rollout status -n {NAMESPACE} deploy/python-toolbox --timeout={TOOLBOX_ROLLOUT_TIMEOUT_SECONDS}s",
                check=False,
            )
        elif WAIT_FOR_TOOLBOX_ROLLOUT and not desired_toolbox_enabled:
            deploy_check = run_kubectl(f"get deploy -n {NAMESPACE} python-toolbox", check=False, quiet=True)
            if deploy_check.returncode != 0:
                print("python-toolbox deployment is absent after disable request.")
            else:
                print("python-toolbox deployment still exists. Re-run the status cell above to inspect current state.")
    else:
        print("Preview mode only. Set APPLY_PYTHON_TOOLBOX_TOGGLE = True to execute Helm from this notebook.")


In [ ]:
TOOLBOX_EXAMPLE_DIR = "/workspace/examples"
EXAMPLE_SCRIPTS = ["langsmith_healthcheck.py", "ollama_smoke.py"]
running_pods = list_running_toolbox_pods()
pod_name = running_pods[0] if running_pods else ""

if pod_name:
    print("Toolbox pod:", pod_name)
    ls_proc = run_kubectl(
        f"exec -n {NAMESPACE} {pod_name} -- sh -lc {shlex.quote(f'ls -1 {TOOLBOX_EXAMPLE_DIR} 2>/dev/null')}",
        check=False,
    )
    available_scripts = {line.strip() for line in ls_proc.stdout.splitlines() if line.strip()}
    if available_scripts:
        print("Available toolbox scripts:", ", ".join(sorted(available_scripts)))

    for script in EXAMPLE_SCRIPTS:
        if available_scripts and script not in available_scripts:
            print(f"Skipping {script}: not found in deployed image. Rebuild/import python-toolbox if needed.")
            continue
        run_kubectl(
            f"exec -n {NAMESPACE} {pod_name} -- python {TOOLBOX_EXAMPLE_DIR}/{script}",
            check=False,
        )
else:
    print("No running toolbox pod found.")
    print("Set PYTHON_TOOLBOX_TOGGLE = 'on' and APPLY_PYTHON_TOOLBOX_TOGGLE = True in the previous cell, then re-run this cell.")


## Pod Memory and CPU Profiling


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def parse_memory_to_mi(mem: str) -> float:
    mem = str(mem).strip()
    if mem.endswith("Gi"):
        return float(mem[:-2]) * 1024.0
    if mem.endswith("Mi"):
        return float(mem[:-2])
    if mem.endswith("Ki"):
        return float(mem[:-2]) / 1024.0
    return float(mem)


top_proc = run_kubectl(f"top pods -n {NAMESPACE} --no-headers", check=False)
lines = [line for line in top_proc.stdout.splitlines() if line.strip()]

rows = []
for line in lines:
    parts = line.split()
    if len(parts) >= 3:
        rows.append({"pod": parts[0], "cpu": parts[1], "memory": parts[2]})

top_df = pd.DataFrame(rows)
display(top_df)

if top_proc.returncode != 0:
    print("kubectl top is unavailable or metrics-server is not ready.")

if not top_df.empty:
    plot_df = top_df.copy()
    plot_df["memory_mi"] = plot_df["memory"].apply(parse_memory_to_mi)
    ax = plot_df.sort_values("memory_mi", ascending=False).plot(
        x="pod", y="memory_mi", kind="bar", figsize=(11, 4), title="Pod Memory Usage (Mi)"
    )
    ax.set_ylabel("Memory (Mi)")
    plt.xticks(rotation=45, ha="right")
    plt.show()


## Correlate Inference With GPU Activity


In [ ]:
import requests
import time

LANGCHAIN_PROXY_PORT_FORWARD_CMD = "kubectl port-forward -n llm-observability svc/langchain-demo 8000:8000"
PROXY_HEALTH_URL = "http://localhost:8000/healthz"
PROXY_CHAT_URL = "http://localhost:8000/ollama/api/chat"

print("GPU before inference:")
run("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", check=False)

payload = {
    "model": "gemma3-1b-it-gguf-local",
    "stream": False,
    "messages": [{"role": "user", "content": "Describe how Kubernetes cronjobs work in 3 lines."}],
}

proxy_ready = False
try:
    health = requests.get(PROXY_HEALTH_URL, timeout=5)
    proxy_ready = health.status_code < 500
except Exception:
    proxy_ready = False

if not proxy_ready:
    print("Skipping GPU correlation inference: langchain-demo proxy is not reachable on localhost:8000")
    print("In another terminal run:")
    print(f"  {LANGCHAIN_PROXY_PORT_FORWARD_CMD}")
else:
    try:
        started = time.perf_counter()
        resp = requests.post(PROXY_CHAT_URL, json=payload, timeout=240)
        elapsed = (time.perf_counter() - started) * 1000
        print("inference status:", resp.status_code, "latency_ms:", round(elapsed, 2))
    except Exception as exc:
        print(f"Inference request skipped/failed: {exc}")

print("GPU after inference:")
run("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", check=False)


## Search for Long-Running Python Processes in Stack Pods


In [ ]:
pods_json = run_kubectl_json(f"get pods -n {NAMESPACE} -o json", check=False)
pod_names = [item.get("metadata", {}).get("name", "") for item in pods_json.get("items", []) if item.get("metadata", {}).get("name")]

if not pod_names:
    print(f"No pods found in namespace: {NAMESPACE}")

for pod in pod_names:
    ps_cmd = (
        "if command -v ps >/dev/null 2>&1; then "
        "(ps -eo pid,etime,comm,args 2>/dev/null || ps -o pid,etime,comm,args 2>/dev/null || ps) "
        "| grep -E 'python|uvicorn|jupyter' | grep -v grep || true; "
        "else echo 'ps command not available in this container'; fi"
    )
    cmd = f"exec -n {NAMESPACE} {pod} -- sh -lc {shlex.quote(ps_cmd)}"
    print("=" * 120)
    print(pod)
    run_kubectl(cmd, check=False)


## Done
You now have repeatable diagnostics for toolbox scripts, memory usage, and GPU-associated inference checks.
